# Entrenamiento del modelo de señas con movimiento (Google Colab)

Equivale a `ml/train_sequence.py` (anexo A.8) para cuando TensorFlow no instala en el computador (Windows, D-020). Usa **uv** con el mismo `ml/pyproject.toml`, así las versiones son idénticas (TensorFlow 2.15.1, tensorflowjs 4.22.0, `GRU(reset_after=False)`).

**Plan de mejora, paso C (D-045).** Entrena **candidatos** en carpetas aparte (`candidato-5/` y `candidato-todas/`). El modelo publicado (`apps/web/public/models/lsc-gestos/`) no se toca: se reemplaza a mano solo si el candidato gana (ver la última celda).

1. Sube `ml/pyproject.toml`, `ml/train_sequence.py`, `ml/train_static.py`, `content/signs.csv` y los `dataset-*.json` (p. ej. `dataset-lsc-54-holistic-v2.json` del importador y los del colector `/admin/gestos`).
2. Ejecuta las celdas en orden. Cada entrenamiento hace validación cruzada **por persona**: nadie se evalúa con un modelo que la vio, y la parada temprana usa otras personas del entrenamiento, nunca las evaluadas.
3. Lee el reporte: acierto por seña **con el umbral 0,70 de la aplicación**, señas «no reconocidas», confusiones y falsas detecciones de NINGUNA.
4. Una seña se publica solo si llega a **≥ 80 %** con personas no vistas y tiene al menos 10 muestras (D-043).

In [ ]:
# uv suele venir en Colab; si no, se instala con su instalador oficial.
!command -v uv || curl -LsSf https://astral.sh/uv/install.sh | sh
import os

os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]
!uv --version

In [ ]:
import os
import shutil

from google.colab import files

print("Sube pyproject.toml, train_sequence.py, train_static.py, signs.csv y tus dataset-*.json")
uploaded = files.upload()
for folder in ("ml/data", "content"):
    os.makedirs(folder, exist_ok=True)
DESTINOS = {
    "pyproject.toml": "ml/pyproject.toml",
    "train_sequence.py": "ml/train_sequence.py",
    "train_static.py": "ml/train_static.py",
    "signs.csv": "content/signs.csv",
}
for name in uploaded:
    if name in DESTINOS:
        shutil.move(name, DESTINOS[name])
    elif name.endswith(".json"):
        shutil.move(name, f"ml/data/{name}")
print(sorted(os.listdir("ml/data")))

## Candidato 1: las mismas 5 señas del modelo publicado

Hola, adiós, por favor, de nada y perdón; el resto de señas del dataset cuenta como NINGUNA. Sirve para comparar **seña por seña** con el modelo publicado (`meta.json`: 89,7 % por persona, sin umbral). No es una comparación perfecta: cambian los datos y las personas; por eso se mira cada seña y no un solo número.

In [ ]:
!MPLBACKEND=Agg uv run --project ml python ml/train_sequence.py --data 'ml/data/*.json' --signs content/signs.csv --out candidato-5 --reports reportes-5 --labels hola adios por-favor de-nada perdon --others-as-none --balance

## Candidato 2: todas las señas del dataset

Todas las señas de los datasets subidos (las 18 de LSC-54 más las que haya grabado el equipo) + NINGUNA, con pesos por clase (`--balance`) porque NINGUNA tiene muchas más muestras que «sí» o «no».

In [ ]:
!MPLBACKEND=Agg uv run --project ml python ml/train_sequence.py --data 'ml/data/*.json' --signs content/signs.csv --out candidato-todas --reports reportes-todas --balance

In [ ]:
import json

from IPython.display import Image, display

for carpeta in ("reportes-5", "reportes-todas"):
    reporte = json.load(open(f"{carpeta}/reporte-gestos.json", encoding="utf-8"))
    print(f"== {carpeta} ==")
    print(f"Acierto con el umbral {reporte['threshold']}: {reporte['accuracy']:.1%}")
    print(f"Falsas detecciones de NINGUNA: {reporte['falseDetections']:.1%}")
    for fila in reporte["classes"]:
        print(
            f"  {fila['label']:>14}: {fila['recall']:.1%} de {fila['samples']}",
            f"· no reconocidas {fila['rejected']} · confusiones {fila['confusedWith']}",
        )
    print("Publicables:", reporte["publishable"])
    display(Image(f"{carpeta}/confusion-gestos.png"))

## Descargar el candidato elegido

Cambia `ELEGIDO` por `candidato-5` o `candidato-todas`. El ZIP trae el modelo y su reporte.

**Antes de publicarlo** (en el computador):
1. Guarda una copia de `apps/web/public/models/lsc-gestos/` (la reversión es volver a copiarla).
2. Si el candidato trae señas que **no** son publicables, vuelve a entrenar solo con las publicables (`--labels … --others-as-none`): una seña que la cámara reconoce mal es peor que una sin cámara.
3. Copia la carpeta **tal cual** (no renombres `group1-shard*.bin`) y corre `npm --prefix apps/web run test -- --run src/camera/holistic/published-model.test.ts`.
4. Prueba con personas que no estén en LSC-54 (D-043) y registra el resultado en `docs/PROGRESS.md`.

In [ ]:
ELEGIDO = "candidato-todas"
reportes = ELEGIDO.replace("candidato", "reportes")
shutil.copytree(reportes, f"{ELEGIDO}-con-reporte/reporte", dirs_exist_ok=True)
shutil.copytree(ELEGIDO, f"{ELEGIDO}-con-reporte/lsc-gestos", dirs_exist_ok=True)
shutil.make_archive(ELEGIDO, "zip", f"{ELEGIDO}-con-reporte")
files.download(f"{ELEGIDO}.zip")